# Fine-Tuning Mistral-7B Locally on Apple Silicon (M-series Mac)

This notebook walks you through fine-tuning **Mistral-7B-Instruct-v0.3** entirely on your local machine using Apple MLX — no cloud, no GPU cluster, no API costs.

**Tested on:** M4 Pro MacBook · 16 GB RAM · Peak memory: ~4.1 GB

## What you will learn
- Why fine-tuning beats prompting for domain-specific tasks
- How to prepare a custom JSONL dataset
- How to run LoRA fine-tuning with `mlx_lm`
- How to fuse adapters and run inference locally

## Prerequisites
```bash
pip install mlx-lm
```
> Requires macOS 14+ and an Apple Silicon chip (M1 and above).

> **Alternative (no-code):** [Unsloth Studio](https://unsloth.ai) provides a UI-based workflow for the same fine-tuning pipeline — no code required.

## Step 1 — Prepare Your Training Dataset

Mistral-7B-Instruct works best with the `messages` format (user/assistant pairs). `mlx_lm` automatically maps this to the `[INST]` prompt template during training.

The dataset below uses **LLM & AI concepts** as the demo domain. Swap it with your own Q&A pairs to specialise the model for any use case.

> **Tip:** Even 10–20 high-quality examples are enough to observe behavioural shifts. For production use, aim for 500+ diverse examples.

In [ ]:
import json
import os

# ------------------------------------------------------------------
# Configuration — edit these paths to match your environment
# ------------------------------------------------------------------
DATA_DIR     = "models/finetune_data"
MODEL_ID     = "mlx-community/Mistral-7B-Instruct-v0.3-4bit"
ADAPTER_PATH = "./models/mistral_adapters"
FUSED_PATH   = "./models/mistral_fused"

os.makedirs(DATA_DIR, exist_ok=True)

# ------------------------------------------------------------------
# Training data
# Replace these examples with your own domain-specific Q&A pairs.
# Format: list of {messages: [{role, content}, ...]} dicts
# ------------------------------------------------------------------
training_data = [
    {
        "messages": [
            {"role": "user",      "content": "What is the difference between fine-tuning and prompt engineering?"},
            {"role": "assistant", "content": "Prompt engineering shapes model behaviour through carefully crafted inputs at inference time — no weights are changed. Fine-tuning updates the model weights on domain-specific data, producing a specialised model that is faster, cheaper, and more accurate for that domain without needing long system prompts."}
        ]
    },
    {
        "messages": [
            {"role": "user",      "content": "What is LoRA and why is it used for fine-tuning?"},
            {"role": "assistant", "content": "LoRA (Low-Rank Adaptation) is a parameter-efficient fine-tuning technique. Instead of updating all model weights, it injects small trainable matrices into the attention layers. This drastically reduces memory usage and training time while retaining most of the performance gain of full fine-tuning."}
        ]
    },
    {
        "messages": [
            {"role": "user",      "content": "How does 4-bit quantization help when running large models locally?"},
            {"role": "assistant", "content": "4-bit quantization compresses model weights from 16-bit floats to 4-bit integers, reducing memory footprint by ~4x with minimal quality loss. A 7B model that would normally require ~14 GB of RAM can run comfortably in ~4 GB, making it feasible on consumer hardware like an M-series MacBook."}
        ]
    },
    {
        "messages": [
            {"role": "user",      "content": "What is the role of the learning rate in LoRA fine-tuning?"},
            {"role": "assistant", "content": "The learning rate controls how aggressively the adapter weights are updated each step. For large models like Mistral-7B, a lower learning rate (e.g. 1e-5) prevents instability and catastrophic forgetting of the base model knowledge. Higher rates train faster but risk overfitting or degraded general capability."}
        ]
    },
    {
        "messages": [
            {"role": "user",      "content": "What is the difference between base and instruct variants of an LLM?"},
            {"role": "assistant", "content": "A base model is pre-trained on raw text and predicts the next token — it has no concept of following instructions. An instruct variant is further fine-tuned (often with RLHF or SFT) on instruction-response pairs, making it follow user prompts conversationally. For fine-tuning on Q&A tasks, instruct models are the better starting point."}
        ]
    },
    {
        "messages": [
            {"role": "user",      "content": "Why run LLMs locally instead of using a cloud API?"},
            {"role": "assistant", "content": "Running LLMs locally offers three main advantages: data privacy (nothing leaves your machine), zero API costs (no per-token billing), and low latency for on-device inference. For sensitive domains like healthcare, legal, or internal enterprise data, local models are often the only viable option."}
        ]
    },
    {
        "messages": [
            {"role": "user",      "content": "What does the adapter fusion step do after LoRA training?"},
            {"role": "assistant", "content": "Fusion merges the trained LoRA adapter weights back into the base model weights, producing a single standalone model file. This eliminates the need to load both the base model and adapter at inference time, simplifying deployment and slightly improving inference speed."}
        ]
    },
    {
        "messages": [
            {"role": "user",      "content": "How many training examples do I need for effective fine-tuning?"},
            {"role": "assistant", "content": "It depends on the task. For concept demonstrations or narrow domain adaptation, 10–50 high-quality examples can show clear behavioural change. For production-grade fine-tuning with consistent output quality, aim for 500–5000 diverse, well-labelled examples. Quality always beats quantity."}
        ]
    },
    {
        "messages": [
            {"role": "user",      "content": "What is the mask-prompt flag in mlx_lm training?"},
            {"role": "assistant", "content": "The --mask-prompt flag tells the trainer to compute loss only on the assistant's response tokens, not on the user prompt tokens. This focuses the model on learning to generate better responses rather than memorising the input questions, which leads to more efficient and accurate fine-tuning."}
        ]
    },
    {
        "messages": [
            {"role": "user",      "content": "Can I fine-tune a model on a MacBook with only 16 GB of RAM?"},
            {"role": "assistant", "content": "Yes. Using a 4-bit quantized model (e.g. Mistral-7B-Instruct-v0.3-4bit) with LoRA and batch size 1, the peak memory usage is around 4 GB on Apple Silicon. This leaves substantial headroom on a 16 GB machine. Apple's unified memory architecture also means the GPU and CPU share the same pool, making it significantly more efficient than traditional setups."}
        ]
    }
]

# ------------------------------------------------------------------
# Write splits to disk
# In production: use separate, non-overlapping train/valid/test sets
# ------------------------------------------------------------------
def write_jsonl(filename: str, data: list) -> None:
    path = os.path.join(DATA_DIR, filename)
    with open(path, "w") as f:
        for entry in data:
            f.write(json.dumps(entry) + "\n")

write_jsonl("train.jsonl", training_data)
write_jsonl("valid.jsonl", training_data)
write_jsonl("test.jsonl",  training_data)

print(f"Dataset written to {DATA_DIR}/ ({len(training_data)} examples)")

## Step 2 — Fine-Tune with LoRA

Runs LoRA training via `mlx_lm`. Key parameters:

| Parameter | Value | Notes |
|---|---|---|
| `--iters` | 500 | Increase for larger datasets |
| `--batch-size` | 1 | Safe for 16 GB RAM |
| `--learning-rate` | 1e-5 | Conservative — preserves base model stability |
| `--mask-prompt` | — | Loss computed on assistant responses only |

> **Runtime:** ~10–15 min on M4 Pro for 500 iterations

In [ ]:
# LoRA fine-tuning — adapters are saved to ADAPTER_PATH
!mlx_lm.lora \
    --model {MODEL_ID} \
    --train \
    --data ./{DATA_DIR} \
    --iters 500 \
    --batch-size 1 \
    --learning-rate 1e-5 \
    --adapter-path {ADAPTER_PATH} \
    --mask-prompt

## Step 3 — Fuse Adapters into the Base Model

Merges the trained LoRA adapter weights into the base model, producing a single standalone model ready for deployment.

In [ ]:
# Fuse LoRA adapters — output is a self-contained model at FUSED_PATH
!mlx_lm.fuse \
    --model {MODEL_ID} \
    --adapter-path {ADAPTER_PATH} \
    --save-path {FUSED_PATH}

## Step 4 — Run Inference on the Fine-Tuned Model

Test the fused model with a prompt from your training domain.

- `--temp 0.0` → deterministic (greedy) output, best for evaluation
- Increase temperature for more varied responses in production use

In [ ]:
# Run inference on the fine-tuned model
# Change the prompt to test your own domain questions
!mlx_lm.generate \
    --model {FUSED_PATH} \
    --prompt "What is LoRA and why is it used for fine-tuning?" \
    --max-tokens 150 \
    --temp 0.0

## Notes & Tips

**Performance (M4 Pro, 16 GB RAM)**
- Peak memory: ~4.1 GB
- Prompt processing: ~72 tokens/sec
- Generation speed: ~23 tokens/sec

**Customisation**
- Swap the `training_data` list in Step 1 with your own Q&A pairs for any domain
- Use separate `train.jsonl` and `valid.jsonl` files in production to monitor overfitting
- To resume training from a checkpoint: add `--resume-adapter-file <path>` to the training command
- Increase `--iters` and lower `--learning-rate` for larger, more diverse datasets

**No-code alternative**
[Unsloth Studio](https://unsloth.ai) provides a UI-based interface for the same LoRA fine-tuning workflow — no code required, same results.

## References
- [mlx-lm on GitHub](https://github.com/ml-explore/mlx-examples/tree/main/llms)
- [Unsloth Studio](https://unsloth.ai)
- [Mistral-7B-Instruct-v0.3 on Hugging Face](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3)
- [MLX documentation](https://ml-explore.github.io/mlx/)
- [LoRA paper](https://arxiv.org/abs/2106.09685)